# **Generate Masks**

This notebook processes long `.tif` time series by splitting them into smaller chunks, segmenting each chunk using a pretrained Cellpose model, and relabeling masks across chunks to ensure consistent tracking of cells over time.

### **Import Libraries**

We begin by importing necessary packages, including:
- Cellpose model API
- `tifffile` for reading/writing TIFF stacks
- `numpy`, `scipy`, and `skimage` for image and mask manipulation
- `cct_utils`, which contains our custom tracking logic

In [ ]:
import os
import tifffile
import numpy as np
from tqdm import tqdm
from cellpose import models
from skimage.measure import regionprops
from scipy.spatial.distance import cdist
cct_utils = __import__('0_cct_utils')

### **Define Paths**

Here we define all relevant paths:
- The `.tif` file to process
- The location of the pretrained Cellpose model
- Where the output (masks) will be saved

In [ ]:
model_name = "cellpose_1746802277.9571602"
model_folder = "ModelAB1"
file_name = 'OUA_140525_cluster2_20min_10i_340ms.tif'

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
raw_path = os.path.normpath(os.path.join(parent_dir, "raw_data", model_folder))
mask_path = os.path.normpath(os.path.join(parent_dir, "masks_tracked", model_folder))
model_path = os.path.normpath(os.path.join(parent_dir, "saved_models", model_folder, "cellpose_train", "models", model_name))

os.makedirs(mask_path, exist_ok=True)

### **Segmentation and Tracking Function**

The `track_Y` function:
- Removes any trailing empty frames
- Uses Cellpose to segment each frame individually
- Uses a custom tracking function (`get_tracked_masks`) to create temporally consistent labels within a chunk

In [ ]:
def track_Y(X, model, diam=None):
    crop_idx = np.argmax(np.mean(X, axis=(1, 2)) == 0)
    X = X[:crop_idx] if crop_idx > 0 else X

    if len(X) == 0:
        return np.zeros_like(X)

    Y = [
        model.eval(np.squeeze(i), diameter=None, channels=[0, 0],
                   flow_threshold=0.8, cellprob_threshold=0.4, do_3D=False)[0]
        for i in np.split(X, X.shape[0])
    ]

    if len(Y) == 0:
        return np.zeros_like(X)

    tracked = cct_utils.get_tracked_masks(masks=np.array(Y))

    return tracked

### **Step 1: Split Time Series into Chunks**

Since long `.tif` files can cause segmentation issues, we split the image into smaller chunks (default: 10 frames each).<br>
This makes processing manageable and improves accuracy.

In [ ]:
def split_tif_stack(full_stack, chunk_size=10):
    return [full_stack[i:i + chunk_size] for i in range(0, len(full_stack), chunk_size)]

### **Step 2: Segment Each Chunk**

Each chunk is independently segmented using our Cellpose model and `track_Y`.<br>
This step creates a label mask stack for each small group of frames.

In [ ]:
def segment_chunks(chunks, model):
    segmented = []
    with tqdm(total=len(chunks), desc="Tracking chunks", unit="chunk", position=0, leave=True) as chunk_bar:
        for chunk in chunks:
            tracked = track_Y(chunk, model, diam=None)
            segmented.append(tracked)
            chunk_bar.update(1)
    return segmented

### **Step 3: Relabel Across Chunks**

To ensure consistent labels across chunk boundaries, we compare the last frame of each chunk with the first frame of the next.<br>
Using centroid proximity, we match similar objects and propagate labels. This helps maintain consistent cell identity across the entire sequence.

In [ ]:
def relabel_chunks(chunks, dist_thresh=10):
    relabeled_chunks = []
    next_label = 1
    prev_labels = {}

    for i, chunk in enumerate(chunks):
        relabeled = np.zeros_like(chunk)

        for t in range(chunk.shape[0]):
            curr = chunk[t]
            props = regionprops(curr)
            curr_centroids = np.array([p.centroid for p in props])
            curr_labels = [p.label for p in props]

            if i == 0 and t == 0:
                for prop in props:
                    relabeled[t][curr == prop.label] = next_label
                    prev_labels[next_label] = prop.centroid
                    next_label += 1
            else:
                new_labels = {}
                if len(curr_centroids) and len(prev_labels):
                    prev_ids = list(prev_labels.keys())
                    prev_centroids = np.array([prev_labels[k] for k in prev_ids])
                    distances = cdist(curr_centroids, prev_centroids)

                    for j, row in enumerate(distances):
                        if np.min(row) < dist_thresh:
                            matched_idx = np.argmin(row)
                            matched_label = prev_ids[matched_idx]
                        else:
                            matched_label = next_label
                            next_label += 1
                        new_labels[curr_labels[j]] = matched_label
                        prev_labels[matched_label] = curr_centroids[j]
                else:
                    for lbl, centroid in zip(curr_labels, curr_centroids):
                        new_labels[lbl] = next_label
                        prev_labels[next_label] = centroid
                        next_label += 1

                for old_lbl, new_lbl in new_labels.items():
                    relabeled[t][curr == old_lbl] = new_lbl

        relabeled_chunks.append(relabeled)

    return np.concatenate(relabeled_chunks, axis=0)

### **Execute the Mask Generation Pipeline**

Now we run all steps:
1. Load the full `.tif` stack
2. Split into chunks
3. Segment each chunk
4. Relabel to preserve tracking
5. Save the final, merged mask stack

In [ ]:
# --- Mask generation pipeline ---
X = tifffile.imread(os.path.join(raw_path, file_name))
model = models.CellposeModel(gpu=True, pretrained_model=model_path)

print("Splitting...")
chunks = split_tif_stack(X, chunk_size=10)

print("Segmenting...")
segmented_chunks = segment_chunks(chunks, model)

print("Relabeling...")
final_masks = relabel_chunks(segmented_chunks)

output_path = os.path.join(mask_path, file_name)
tifffile.imwrite(output_path, final_masks, imagej=True, metadata={'axes': 'TYX'})
print(f"Final relabeled masks saved to: {output_path}")